# 01 — CV Data Prep: unify CarDD + VehiDE into one YOLO dataset

**Run on Kaggle** (the ~5 GB of images already live there — no local download needed).

Attach these two Kaggle datasets to the notebook:
- `gabrielfcarvalho/cardd-with-yolo-annotations-images-labels` — CarDD, **already YOLO format** (6 damage classes)
- `hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection` — VehiDE, COCO-style (8 classes)

**Goal:** one combined YOLO dataset with a single unified damage-category schema, ready for YOLOv8 fine-tuning in notebook 02. Nothing is dropped silently — any unmapped source class is printed and halts the merge.

---
### Unified damage schema (6 classes)
We collapse both sources onto CarDD's 6 well-defined categories, since they are the pricing-relevant damage types:

| id | class | maps from |
|----|-------|-----------|
| 0 | dent | dent / dented / deformation |
| 1 | scratch | scratch / scratches |
| 2 | crack | crack / cracked |
| 3 | glass_shatter | glass shatter / broken glass / windshield |
| 4 | lamp_broken | lamp broken / broken lamp / headlight |
| 5 | tire_flat | tire flat / flat tire |


In [ ]:
import os, glob, json, shutil
from pathlib import Path
from collections import Counter

# Kaggle mounts inputs read-only under /kaggle/input/<slug>
CARDD = Path("/kaggle/input/cardd-with-yolo-annotations-images-labels")
VEHIDE = Path("/kaggle/input/vehide-dataset-automatic-vehicle-damage-detection")
OUT = Path("/kaggle/working/combined")   # writable
for p in [CARDD, VEHIDE]:
    print(p, "exists:", p.exists())
    if p.exists():
        for d in list(p.rglob('*'))[:0]: pass
        print("  top:", sorted({q.name for q in p.glob('*')}))

### Unified schema + source→unified remaps
Edit the remap dicts only if the printed source class names differ from what's assumed below.

In [ ]:
UNIFIED = ["dent","scratch","crack","glass_shatter","lamp_broken","tire_flat"]
U = {name:i for i,name in enumerate(UNIFIED)}

def norm(s): return str(s).strip().lower().replace('-',' ').replace('_',' ')

# CarDD's own class order (from its data.yaml) — verified at runtime below.
CARDD_REMAP = {
    "dent":"dent","scratch":"scratch","crack":"crack",
    "glass shatter":"glass_shatter","lamp broken":"lamp_broken","tire flat":"tire_flat",
}
# VehiDE categories → unified. Anything not here is reported (never dropped silently).
VEHIDE_REMAP = {
    "dent":"dent","dented":"dent","deformation":"dent",
    "scratch":"scratch","scratches":"scratch",
    "crack":"crack","cracked":"crack",
    "glass shatter":"glass_shatter","broken glass":"glass_shatter","windshield damage":"glass_shatter",
    "lamp broken":"lamp_broken","broken lamp":"lamp_broken","headlight":"lamp_broken",
    "tire flat":"tire_flat","flat tire":"tire_flat",
}
print("unified:", U)

### 1. Discover CarDD's YOLO layout and class names

In [ ]:
import yaml
cardd_yaml = next(iter(glob.glob(str(CARDD/'**/data.yaml'), recursive=True)), None)
print("CarDD data.yaml:", cardd_yaml)
cardd_names = None
if cardd_yaml:
    y = yaml.safe_load(open(cardd_yaml))
    cardd_names = y.get('names')
    print("CarDD class order:", cardd_names)
# sanity: every CarDD class must have a unified mapping
if cardd_names:
    unmapped = [c for c in cardd_names if CARDD_REMAP.get(norm(c)) not in U]
    assert not unmapped, f"Unmapped CarDD classes: {unmapped}"
    print("CarDD classes all mapped ✓")

### 2. Copy CarDD (YOLO→YOLO, remap class ids)

In [ ]:
def cardd_class_to_unified(old_id):
    src = norm(cardd_names[int(old_id)])
    return U[CARDD_REMAP[src]]

def copy_yolo_split(img_glob, lbl_dir, split, id_fn, tag):
    (OUT/'images'/split).mkdir(parents=True, exist_ok=True)
    (OUT/'labels'/split).mkdir(parents=True, exist_ok=True)
    n = 0
    for img in glob.glob(img_glob, recursive=True):
        stem = Path(img).stem
        lbl = Path(lbl_dir)/f"{stem}.txt"
        dst_img = OUT/'images'/split/f"{tag}_{Path(img).name}"
        shutil.copy(img, dst_img)
        lines=[]
        if lbl.exists():
            for ln in open(lbl):
                p = ln.split()
                if len(p)>=5:
                    p[0]=str(id_fn(p[0])); lines.append(' '.join(p))
        (OUT/'labels'/split/f"{tag}_{stem}.txt").write_text('\n'.join(lines))
        n+=1
    return n

# CarDD ships train/val/test splits; adjust globs to the discovered structure.
root = Path(cardd_yaml).parent if cardd_yaml else CARDD
for split in ['train','val']:
    imgs = str(root/'images'/split/'**/*.*')
    lbls = root/'labels'/split
    if glob.glob(imgs, recursive=True):
        c = copy_yolo_split(imgs, lbls, split, cardd_class_to_unified, 'cardd')
        print(f"CarDD {split}: {c} images")

### 3. Convert VehiDE (COCO→YOLO, remap categories)
Halts if any VehiDE category is unmapped — inspect the print and extend `VEHIDE_REMAP`.

In [ ]:
def coco_to_yolo(coco_json, img_root, split, tag='vehide'):
    data = json.load(open(coco_json))
    cats = {c['id']: norm(c['name']) for c in data['categories']}
    unmapped = sorted({n for n in cats.values() if n not in VEHIDE_REMAP})
    assert not unmapped, f"Unmapped VehiDE categories: {unmapped} -- extend VEHIDE_REMAP"
    imgs = {im['id']: im for im in data['images']}
    from collections import defaultdict
    anns = defaultdict(list)
    for a in data['annotations']:
        anns[a['image_id']].append(a)
    (OUT/'images'/split).mkdir(parents=True, exist_ok=True)
    (OUT/'labels'/split).mkdir(parents=True, exist_ok=True)
    n=0
    for iid, im in imgs.items():
        src = Path(img_root)/im['file_name']
        if not src.exists(): continue
        W,H = im['width'], im['height']
        lines=[]
        for a in anns.get(iid, []):
            x,y,w,h = a['bbox']
            cx,cy = (x+w/2)/W, (y+h/2)/H
            uid = U[VEHIDE_REMAP[cats[a['category_id']]]]
            lines.append(f"{uid} {cx:.6f} {cy:.6f} {w/W:.6f} {h/H:.6f}")
        shutil.copy(src, OUT/'images'/split/f"{tag}_{src.name}")
        (OUT/'labels'/split/f"{tag}_{src.stem}.txt").write_text('\n'.join(lines))
        n+=1
    return n

# Locate VehiDE COCO json(s) + image roots (structure printed in cell 2), then:
for cj in glob.glob(str(VEHIDE/'**/*.json'), recursive=True):
    name = Path(cj).stem.lower()
    split = 'val' if ('val' in name or 'test' in name) else 'train'
    img_root = Path(cj).parent  # adjust if images sit in a sibling folder
    try:
        c = coco_to_yolo(cj, img_root, split); print(f"VehiDE {cj} -> {split}: {c} images")
    except AssertionError as e:
        print("STOP:", e)

### 4. Write `data.yaml` and report class balance

In [ ]:
(OUT).mkdir(parents=True, exist_ok=True)
yaml_txt = f'''path: {OUT}
train: images/train
val: images/val
nc: {len(UNIFIED)}
names: {UNIFIED}
'''
(OUT/'data.yaml').write_text(yaml_txt); print(yaml_txt)

cnt = Counter()
for lf in glob.glob(str(OUT/'labels/**/*.txt'), recursive=True):
    for ln in open(lf):
        if ln.strip(): cnt[UNIFIED[int(ln.split()[0])]] += 1
print("Unified class instance counts:")
for k in UNIFIED: print(f"  {k:14s} {cnt[k]}")
print("Total images:", len(glob.glob(str(OUT/'images/**/*.*'), recursive=True)))

### Output
`/kaggle/working/combined/` — a single YOLO dataset (`images/`, `labels/`, `data.yaml`) over the unified 6-class schema.
**Save it as a Kaggle dataset output**, then attach it to notebook `02_yolov8_finetune_cardd_vehide.ipynb` for training. Record the printed class-balance numbers in `docs/ARCHITECTURE.md`.